# 03 — Benchmark: SGT-QAT Checkpoint as vLLM Drafter

Loads `checkpoints/qwen3-1.7b-sgt-qat/` (exported by notebook 01, PPL 15.91, 68.0%
corrected recovery — see `docs/findings.md`) as the draft model in vLLM's
speculative-decode config (`method="draft_model"`), targeting Qwen3-8B, and measures
it with the same `notebooks/common/bench_utils.py` harness as notebook 02's EAGLE-3
baseline — same low-concurrency (sequential single-request) methodology, so numbers
are directly comparable.

**Checkpoint source**: Google Drive (`MyDrive/sgt-qat-draft-checkpoints/qwen3-1.7b-sgt-qat/`),
not git — see `docs/context.md` "Checkpoint storage" for why. This notebook mounts
Drive and copies it to local disk before loading (don't read straight off the Drive
mount — flaky mid-large-file-read).

**Not yet run.** Depends on notebook 02 having been (re-)run with the current harness
first, so this notebook's Compare step has real no-spec/EAGLE-3 numbers to load.</cell id="cell-0">


## Setup

In [ ]:
import os
REPO_NAME = 'sgt-qat-draft'
if not os.path.isdir(REPO_NAME):
    !git clone https://github.com/Resh19S/sgt-qat-draft.git
%cd {REPO_NAME}
# Existing clones don't auto-update -- pull explicitly so this session has the
# current bench_utils.py etc.
!git pull

!pip install -q vllm

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/sgt-qat-draft-checkpoints/qwen3-1.7b-sgt-qat')
LOCAL_CHECKPOINT = Path('checkpoints/qwen3-1.7b-sgt-qat')

if not LOCAL_CHECKPOINT.exists():
    LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    !cp -r {str(DRIVE_CHECKPOINT)} {str(LOCAL_CHECKPOINT)}

# Sanity check: local copy should match the 1.2GB validated in notebook 01. If this
# comes back much smaller, the copy truncated (see docs/logs.md 2026-07-23 -- this
# already happened once due to Drive being full) -- don't proceed to loading it into
# vLLM until this looks right.
!du -sh {str(LOCAL_CHECKPOINT)}

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR / 'notebooks' / 'common'))

import bench_utils
import torch

assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
TARGET_MODEL = 'Qwen/Qwen3-8B'
SGT_QAT_DRAFTER = str(LOCAL_CHECKPOINT)  # local path, method="draft_model" (not a HF repo id like EAGLE-3)
NUM_SPECULATIVE_TOKENS = 3   # must match notebook 02's value for a fair comparison
MAX_TOKENS = 256             # must match notebook 02's value
NUM_PROMPTS = 80             # IMPORTANT: set this to whatever notebook 02 actually used --
                              # comparisons are only meaningful if all three conditions
                              # (no-spec, eagle3, draft_model) ran the same prompt count.

# Same placeholder prompts as notebook 02 -- keep identical across notebooks so the
# three conditions are comparable. Still a smoke-test set, not a real benchmark
# dataset (see notebook 02's TODO on swapping to e.g. mt-bench).
PROMPTS = [
    "Explain the difference between speculative decoding and beam search.",
    "Write a short function in Python that reverses a linked list.",
    "Summarize the plot of Pride and Prejudice in three sentences.",
] * (NUM_PROMPTS // 3 + 1)
PROMPTS = PROMPTS[:NUM_PROMPTS]

## Run: SGT-QAT drafter

In [ ]:
speculative_config_sgt_qat = {
    'method': 'draft_model',
    'model': SGT_QAT_DRAFTER,
    'num_speculative_tokens': NUM_SPECULATIVE_TOKENS,
}

# The EAGLE-3 run in notebook 02 already used 38.60GiB of this A100's 40GiB just for
# the target model + a small EAGLE-3 head. Our drafter is a full 1.7B model (much
# bigger than EAGLE-3's lightweight head) loaded alongside the same 8B target --
# real OOM risk on load. vLLM defaults to reserving KV cache sized for
# max_model_len=40960 (Qwen3-8B's full context) regardless of actual usage; capping
# it here frees real memory for the extra model since our prompts/outputs are short.
result_sgt_qat = bench_utils.run_benchmark(
    run_name='sgt_qat_drafter',
    target_model=TARGET_MODEL,
    prompts=PROMPTS,
    speculative_config=speculative_config_sgt_qat,
    max_tokens=MAX_TOKENS,
    llm_kwargs={'max_model_len': 4096},
)
bench_utils.save_result(result_sgt_qat, results_dir=REPO_DIR / 'results')
result_sgt_qat

## Compare against notebook 02's baselines

Loads the most recent `no_spec_decode_*.json` and `baseline_eagle3_*.json` from
`results/` (i.e. whatever notebook 02 was last run with) and puts all three
conditions side by side. Assumes notebook 02 was (re-)run with the current harness
(`gpu_memory_used_bytes` field, sequential single-request methodology) -- if it
wasn't, re-run notebook 02 first or this will error on old-format JSON.

In [ ]:
import json


def _load_latest(results_dir: Path, run_name_prefix: str) -> bench_utils.BenchResult:
    matches = sorted((results_dir).glob(f"{run_name_prefix}_*.json"))
    if not matches:
        raise FileNotFoundError(
            f"No results/{run_name_prefix}_*.json found -- run notebook 02 first."
        )
    latest = matches[-1]
    data = json.loads(latest.read_text())
    return bench_utils.BenchResult(**data)


results_dir = REPO_DIR / 'results'
result_no_spec = _load_latest(results_dir, 'no_spec_decode')
result_eagle3 = _load_latest(results_dir, 'baseline_eagle3')

bench_utils.summarize([result_no_spec, result_eagle3, result_sgt_qat])

print("\nOnce these numbers look sane, transcribe them into docs/findings.md (methods +")
print("numbers) and note anything surprising in docs/logs.md. This is Phase 4 territory")
print("once all three conditions are backed by real (non-placeholder) prompts.")